In [ ]:
import json, glob, os, tarfile, ast
D = os.path.dirname(glob.glob('/kaggle/input/**/tasks.jsonl', recursive=True)[0])
tasks = [json.loads(l) for l in open(f'{D}/tasks.jsonl')]
out = []
for repo in ('fastapi/fastapi', 'encode/httpx'):
    t = next(x for x in tasks if x['repo'] == repo); key = repo.split('/')[-1]
    nodes = json.load(open(glob.glob(f"{D}/graphs/{key}_{t['base_commit']}.json")[0]))['nodes']
    gshort = {str(n.get('name', '')).split('.')[-1] for n in nodes}
    root = f'/tmp/u_{key}'; os.makedirs(root, exist_ok=True)
    with tarfile.open(glob.glob(f"{D}/snapshots/{t['instance_id']}.tgz")[0]) as tf: tf.extractall(root)
    asyn, syn = set(), set()
    for p in glob.glob(f'{root}/**/*.py', recursive=True):
        try: tree = ast.parse(open(p, errors='ignore').read())
        except Exception: continue
        for nd in ast.walk(tree):
            (asyn if isinstance(nd, ast.AsyncFunctionDef) else syn if isinstance(nd, ast.FunctionDef) else set()).add(nd.name) if isinstance(nd, (ast.AsyncFunctionDef, ast.FunctionDef)) else None
    only_async = sorted(asyn - syn)
    found = [n for n in only_async if n in gshort]
    out.append(dict(repo=repo, async_only_names=len(only_async), found_in_graph=len(found), examples_found=found[:8], examples_missing=[n for n in only_async if n not in gshort][:8]))
    print(out[-1], flush=True)
json.dump(out, open('/kaggle/working/async_unique.json', 'w'), indent=1)
